# ml_models.ipynb — Predicting sleepiness from resting-state EEG
BMEN600 · Group 6 · predictive extension of the midterm plan

The midterm plan asked *"is T/A associated with sleepiness?"* (statistics). This notebook asks *"how well can EEG features **predict** sleepiness in a person the model has never seen?"* (machine learning).

| Task | Target | Type | Unit of prediction |
|---|---|---|---|
| **A** | KSS (1–9) | regression | recording (168) |
| **B** | Sleepy vs alert (KSS ≥ 6) | classification | recording (168) |
| **C** | Sleepier or less sleepy than *own* average | classification | recording (EEG centred on the person's own mean) |
| **D** | Excessive daytime sleepiness (ESS > 10) | classification | participant (56) |

**Data:** `clean_data/sleepiness_eeg_clean.csv` — OpenNeuro ds004148, eyes-closed resting EEG, 56 participants × 3 sessions = 168 recordings (4 participants with missing KSS removed). Columns used: `participant_id`, `session`, `KSS`, `ESS_avg`, `theta`, `alpha`, `theta_alpha_ratio`.

Two feature sets are compared for every task: **T/A only** (the plan's primary biomarker) vs **theta + alpha** (does splitting the bands help?).

**Evaluation rules that matter**
- Every person has up to 3 recordings, and sessions 1 and 2 are only 90 min apart, so recordings from one person are near-duplicates. All cross-validation is **grouped by participant** (no person in both train and test). Random row-wise CV would give inflated scores.
- Hyper-parameters are tuned in an **inner** grouped CV that only sees the training participants (nested CV).
- Every model is compared with a **baseline** that ignores EEG. A model only counts if it beats the baseline, with a participant-bootstrap CI.

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.base import clone
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import (RandomForestRegressor, RandomForestClassifier,
                              GradientBoostingRegressor, GradientBoostingClassifier)
from sklearn.svm import SVR, SVC
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.metrics import (roc_auc_score, balanced_accuracy_score, mean_absolute_error,
                             mean_squared_error, r2_score, roc_curve)
warnings.filterwarnings("ignore")
os.makedirs("results", exist_ok=True); os.makedirs("figures", exist_ok=True)

# ---- settings ----
# Reads the cleaned data straight from the team repo, so it runs in Colab with nothing to upload.
# If you cloned the repo instead, the local copy is used.
LOCAL_PATH = "clean_data/sleepiness_eeg_clean.csv"
GITHUB_URL = ("https://raw.githubusercontent.com/BilodeauS/TheWalkers_ClassProject_TheWalkers-Group6-/"
              "refs/heads/main/clean_data/sleepiness_eeg_clean.csv")
DATA_PATH = LOCAL_PATH if os.path.exists(LOCAL_PATH) else GITHUB_URL
COLS = dict(participant="participant_id", session="session", kss="KSS", ess="ESS_avg",
            theta="theta", alpha="alpha", ta="theta_alpha_ratio")   # column names in sleepiness_eeg_clean.csv
SLEEPY_THRESHOLD = 6      # Task B: KSS >= 6 ("some signs of sleepiness") -> 82 of 168 recordings (49%), a balanced split
ESS_CUTOFF = 10           # Task D
OUTER_K, REPEATS = 8, 3   # grouped CV folds x repeats (raise REPEATS for stabler numbers)
N_BOOT = 1000
N_JOBS = -1

## 1. Load and prepare (same cleaning as the stats notebook)

In [ ]:
raw = pd.read_csv(DATA_PATH)
df = pd.DataFrame({"pid": raw[COLS["participant"]], "session": raw[COLS["session"]],
                   "KSS": raw[COLS["kss"]], "ESS": raw[COLS["ess"]],
                   "theta": raw[COLS["theta"]], "alpha": raw[COLS["alpha"]]})
df["ta"] = raw[COLS["ta"]] if COLS.get("ta") and COLS["ta"] in raw else df.theta / df.alpha
for c in ["theta", "alpha", "ta"]:
    df["log_" + c] = np.log10(df[c])
df = df.dropna(subset=["KSS", "log_ta", "log_theta", "log_alpha"]).copy()
df = df.sort_values(["pid", "session"]).reset_index(drop=True)

# person-centred EEG (Task C) and person-level table (Task D)
for c in ["log_ta", "log_theta", "log_alpha"]:
    df[c + "_c"] = df[c] - df.groupby("pid")[c].transform("mean")
df["KSS_within"] = df.KSS - df.groupby("pid").KSS.transform("mean")
df["ESS_trait"] = df.groupby("pid").ESS.transform("mean")
pl = df.groupby("pid").agg(log_ta=("log_ta", "mean"), log_theta=("log_theta", "mean"),
                           log_alpha=("log_alpha", "mean"), ESS_trait=("ESS_trait", "first")).reset_index()

print(f"Loaded {DATA_PATH}")
print(f"{df.pid.nunique()} participants, {len(df)} recordings")
assert df.pid.nunique() == 56 and len(df) == 168, "unexpected data size - check you are using clean_data/sleepiness_eeg_clean.csv"
print("KSS distribution:", df.KSS.value_counts().sort_index().to_dict())
print(f"Task B positives (KSS >= {SLEEPY_THRESHOLD}): {(df.KSS >= SLEEPY_THRESHOLD).mean():.0%} of recordings")
print(f"Task D positives (ESS > {ESS_CUTOFF}): {(pl.ESS_trait > ESS_CUTOFF).sum()} of {len(pl)} participants")

## 2. Models and evaluation machinery
Models are deliberately simple: with ~56 people and 1–3 features, flexible models mostly overfit. Ridge / logistic regression is the sensible default; random forest, gradient boosting and an RBF-SVM are there to check whether non-linearity helps.

In [ ]:
def get_models(kind):
    if kind == "reg":
        return {
          "Baseline (train mean)": (DummyRegressor(), {}),
          "Ridge": (make_pipeline(StandardScaler(), Ridge()), {"ridge__alpha": [0.1, 1, 10, 100]}),
          "Random forest": (RandomForestRegressor(n_estimators=150, random_state=0),
                            {"max_depth": [2, 4], "min_samples_leaf": [5, 10]}),
          "Gradient boosting": (GradientBoostingRegressor(learning_rate=0.05, random_state=0),
                                {"n_estimators": [50, 100], "max_depth": [1, 2]}),
          "SVR (RBF)": (make_pipeline(StandardScaler(), SVR()), {"svr__C": [0.1, 1, 10]}),
        }
    return {
      "Baseline (no EEG)": (None, {}),
      "Logistic regression": (make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=1000)),
                              {"logisticregression__C": [0.01, 0.1, 1, 10]}),
      "Random forest": (RandomForestClassifier(n_estimators=150, class_weight="balanced", random_state=0),
                        {"max_depth": [2, 4], "min_samples_leaf": [5, 10]}),
      "Gradient boosting": (GradientBoostingClassifier(learning_rate=0.05, random_state=0),
                            {"n_estimators": [50, 100], "max_depth": [1, 2]}),
      "SVM (RBF)": (make_pipeline(StandardScaler(), SVC(probability=True, class_weight="balanced", random_state=0)),
                    {"svc__C": [0.1, 1, 10]}),
    }

def group_folds(groups, k, seed):
    # assign whole participants to folds; different shuffle per repeat
    ug = np.unique(groups); np.random.default_rng(seed).shuffle(ug)
    fold_of = {g: i % k for i, g in enumerate(ug)}
    f = np.array([fold_of[g] for g in groups])
    for i in range(k):
        yield np.where(f != i)[0], np.where(f == i)[0]

def oof_predict(est, grid, X, y, groups, kind):
    '''Out-of-participant predictions, averaged over REPEATS grouped CV runs. Tuning is nested.'''
    if est is None:                       # classification baseline: constant score -> AUC = BA = 0.5 by definition
        return np.full(len(y), 0.5)
    k = min(OUTER_K, len(np.unique(groups)))
    P = np.zeros((REPEATS, len(y)))
    scoring = "neg_mean_absolute_error" if kind == "reg" else "roc_auc"
    for r in range(REPEATS):
        for tr, te in group_folds(groups, k, seed=r):
            if kind == "clf" and len(np.unique(y[tr])) < 2:
                P[r, te] = y[tr][0]; continue
            m = clone(est)
            if grid:
                m = GridSearchCV(m, grid, cv=GroupKFold(3), scoring=scoring, n_jobs=N_JOBS, error_score=0.0 if kind == "clf" else np.nan)
                m.fit(X[tr], y[tr], groups=groups[tr])
            else:
                m.fit(X[tr], y[tr])
            P[r, te] = m.predict(X[te]) if kind == "reg" else m.predict_proba(X[te])[:, 1]
    return P.mean(0)

def metrics(y, p, kind):
    if kind == "reg":
        return dict(R2=r2_score(y, p), MAE=mean_absolute_error(y, p),
                    RMSE=mean_squared_error(y, p) ** 0.5,
                    Spearman=stats.spearmanr(y, p)[0] if np.std(p) > 0 else 0.0)
    thr = y.mean()                        # threshold at class prevalence so balanced accuracy is meaningful
    return dict(AUC=roc_auc_score(y, p) if len(np.unique(y)) > 1 else np.nan,
                BalAcc=balanced_accuracy_score(y, p >= thr) if np.ptp(p) > 0 else 0.5)

def boot_ci(y, p, groups, kind, key, n=N_BOOT, seed=0):
    g = np.random.default_rng(seed)
    idx = {u: np.where(groups == u)[0] for u in np.unique(groups)}
    ids = np.array(list(idx)); vals = []
    for _ in range(n):
        s = np.concatenate([idx[i] for i in g.choice(ids, len(ids))])
        if kind == "clf" and len(np.unique(y[s])) < 2: continue
        vals.append(metrics(y[s], p[s], kind)[key])
    return np.nanpercentile(vals, [2.5, 97.5])

In [ ]:
FEATURE_SETS = {"T/A only": ["log_ta"], "theta + alpha": ["log_theta", "log_alpha"]}
RESULTS, PREDS = [], {}

def run_task(task, data, y, kind, suffix="", groups_col="pid"):
    groups = data[groups_col].values
    key = "R2" if kind == "reg" else "AUC"
    for fs_name, fs in FEATURE_SETS.items():
        X = data[[c + suffix for c in fs]].values
        for mname, (est, grid) in get_models(kind).items():
            if kind == "clf" and est is None and fs_name != "T/A only":
                continue                      # baseline ignores features; report once
            p = oof_predict(est, grid, X, y, groups, kind)
            mt = metrics(y, p, kind); lo, hi = boot_ci(y, p, groups, kind, key)
            RESULTS.append(dict(task=task, features=("-" if est is None or mname.startswith("Baseline") else fs_name),
                                model=mname, **mt, ci_lo=lo, ci_hi=hi, n=len(y)))
            PREDS[(task, fs_name, mname)] = (y, p)
    print(f"done: {task}")

## 3. Task A — predict KSS (regression)

In [ ]:
run_task("A: KSS regression", df, df.KSS.values.astype(float), "reg")

## 4. Task B — sleepy vs alert (KSS ≥ threshold)

In [ ]:
yB = (df.KSS >= SLEEPY_THRESHOLD).astype(int).values
run_task(f"B: sleepy (KSS>={SLEEPY_THRESHOLD})", df, yB, "clf")

## 5. Task C — sleepier than your *own* usual? (within-person)
EEG features are centred on each person's own mean, so between-person differences in baseline EEG are removed (the same idea as Figure 1 in the plan).
**Deployment caveat:** this assumes a personal baseline exists (e.g., a calibration recording). The centring uses only EEG, never KSS, so there is no label leakage.

In [ ]:
dC = df[df.KSS_within != 0].copy()
yC = (dC.KSS_within > 0).astype(int).values
print(f"Task C: {len(dC)} recordings, {yC.mean():.0%} above own mean")
run_task("C: within-person direction", dC, yC, "clf", suffix="_c")

## 6. Task D — trait: excessive daytime sleepiness from average EEG (participant level)

In [ ]:
yD = (pl.ESS_trait > ESS_CUTOFF).astype(int).values
print(f"Task D: n={len(pl)} participants, {yD.sum()} positive")
run_task(f"D: ESS>{ESS_CUTOFF}", pl, yD, "clf")

## 7. Results

In [ ]:
res = pd.DataFrame(RESULTS)
res.round(3).to_csv("results/ml_results.csv", index=False)
pd.set_option("display.width", 200)
for t, g in res.groupby("task", sort=True):
    print("\n" + t)
    print(g.drop(columns=["task"]).round(3).to_string(index=False))

In [ ]:
# Summary figure: primary metric with participant-bootstrap 95% CI, per task
tasks = res.task.unique()
fig, axes = plt.subplots(1, len(tasks), figsize=(4.2 * len(tasks), 4), sharey=False)
colors = {"T/A only": "#08519c", "theta + alpha": "#e6550d", "-": "#999999"}
for ax, t in zip(np.atleast_1d(axes), tasks):
    g = res[res.task == t].reset_index(drop=True)
    key = "R2" if t.startswith("A") else "AUC"
    ax.barh(range(len(g)), g[key], color=[colors[f] for f in g.features],
            xerr=[g[key] - g.ci_lo, g.ci_hi - g[key]], capsize=2)
    ax.set_yticks(range(len(g)))
    ax.set_yticklabels([f"{m}" + (f" [{f}]" if f != "-" else "") for m, f in zip(g.model, g.features)], fontsize=6.5)
    ax.invert_yaxis(); ax.axvline(0 if key == "R2" else 0.5, color="k", lw=.8, ls="--")
    ax.set_title(t, fontsize=9); ax.set_xlabel(key + (" (out-of-participant)" if key == "R2" else ""))
plt.tight_layout(); plt.savefig("figures/ml_summary.png", dpi=250); plt.show()

In [ ]:
# Best model per classification task: ROC;  best regression model: predicted vs actual
best = res[res.model.str.startswith("Baseline") == False]
fig, ax = plt.subplots(1, 2, figsize=(9.5, 4))
bA = best[best.task.str.startswith("A")].sort_values("R2").iloc[-1]
y, p = PREDS[(bA.task, bA.features, bA.model)]
jit = np.random.default_rng(0).normal(0, .06, len(y))
ax[0].scatter(y + jit, p, s=14, alpha=.5); ax[0].plot([1, 9], [1, 9], "k--", lw=.8)
ax[0].set(xlabel="Actual KSS", ylabel="Predicted KSS", title=f"Task A best: {bA.model} [{bA.features}]  R\u00b2={bA.R2:.2f}")
for t in [x for x in tasks if not x.startswith("A")]:
    b = best[best.task == t].sort_values("AUC").iloc[-1]
    y, p = PREDS[(t, b.features, b.model)]
    fpr, tpr, _ = roc_curve(y, p); ax[1].plot(fpr, tpr, label=f"{t[:1]}: {b.model} (AUC {b.AUC:.2f})")
ax[1].plot([0, 1], [0, 1], "k--", lw=.8); ax[1].set(xlabel="False positive rate", ylabel="True positive rate", title="Best classifier per task")
ax[1].legend(fontsize=7, frameon=False)
plt.tight_layout(); plt.savefig("figures/ml_best_models.png", dpi=250); plt.show()

## 8. What drives the predictions (descriptive only)
Standardised ridge coefficients for Task A fit on **all** data. These describe the fitted relationship; they are *not* a performance estimate.

In [ ]:
for fs_name, fs in FEATURE_SETS.items():
    X = df[fs].values; m = make_pipeline(StandardScaler(), Ridge(alpha=10)).fit(X, df.KSS.values)
    print(fs_name, dict(zip(fs, m[-1].coef_.round(3))), "(KSS points per 1 SD)")

## 9. How to read and report this
- **Compare against the baseline row, not against zero.** For regression R² ≤ 0 means no better than guessing the training-set mean KSS; for classification AUC 0.5 is chance. If the CI overlaps the baseline, say so; a null predictive result is a legitimate finding.
- **Expect modest numbers.** The plan's within-person correlation is r ≈ 0.29 (≈ 8% of variance), which caps how good a recording-level KSS predictor can be from channel-averaged T/A.
- **Report the T/A-only vs theta + alpha comparison**, since it directly tests whether the single ratio loses information.
- **Limits to state in the report:** 56 people (168 recordings); one dataset, one lab; KSS is self-report; only global band power was used (no channel/regional, delta, beta or spindle features); no external validation set; Task C assumes a personal baseline; sessions 1 and 2 share time of day.
- **Possible extensions if time allows:** per-channel or regional band power (needs re-running `consolidate_raw_data.ipynb`), adding delta/beta, and session as a feature (time-of-day confound).